## GenZ Chatbot Assistant

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import pandas as pd

/Users/sanka/Projects/genz_chatbot/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sanka/Projects/genz_chatbot/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Pull data from dataset

In [3]:
# Load the dataset
df = pd.read_csv('../data/genz_slang.csv')

# Display the first few rows of the dataset
print(df.head())


     Slang                                        Description  \
0        W                                  Shorthand for win   
1        L                          Shorthand for loss/losing   
2  L+ratio  Response to a comment or action on the interne...   
3     Dank                  excellent or of very high quality   
4   Cheugy  Derogatory term for Millennials. Used when mil...   

                                             Example  \
0                          Got the job today, big W!   
1           I forgot my wallet at home, that’s an L.   
2  Your tweet got 5 likes and 100 replies calling...   
3                              That meme is so dank!   
4  That phrase is so cheugy, no one says that any...   

                                             Context  
0  Typically used in conversations to celebrate s...  
1  Often used when referring to a failure or mish...  
2  Popularized on social media platforms to signi...  
3  Commonly used in internet slang to refer to me...

### Load OPENAI_API_KEY

In [4]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


### Create a simple chat without the DB

In [5]:
system_message = """
You are a helpful assistant for a GenZ chatbot.
Be friendly and engaging.
Give it an example of how to use the slang word, only when nessesary.
"""

In [6]:
def chat(message, history):
    # add user message to history
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    # Get the response from the model
    response  = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=100
    )

    return response.choices[0].message.content

In [7]:
# test the chat function
gr.ChatInterface(fn=chat, type="messages").launch()

Running on local URL:  http://127.0.0.1:7872

To create a public link, set `share=True` in `launch()`.


### Add Tools

write a function, and have the LLM call that function as part of its response.

In [28]:
slang = {
    "bussin": "Being very successful or popular.",
    "lit": "Excellent or impressive.",
    "goals": "Aspirations or ambitions.",
    "spicy": "Very good or enjoyable.",
    "lit": "Excellent or impressive.",
    "goals": "Aspirations or ambitions.",
    "spicy": "Very good or enjoyable.",
}

In [29]:
def get_slang_definition(slang_word):
    print(f"Getting definition for {slang_word}")
    slang_definition = slang.get(slang_word.lower(), "No definition found for this slang word.")
    return f"{slang_word} means {slang_definition}"

In [30]:
get_slang_definition("bussin")

Getting definition for bussin


'bussin means Being very successful or popular.'

In [31]:
# There's a particular dictionary structure that's required to describe our function:

slang_function = {
    "name": "get_slang_definition",
    "description": "Get the definition of a slang word",
    "parameters": {
        "type": "object",
        "properties": {
            "slang_word": {
                "type": "string",
                "description": "The slang word to get the definition of"
            }
        },
        "required": ["slang_word"],
        "additionalProperties": False,
    },
}




In [36]:
tools = [{
    "type": "function",
    "function": slang_function,
}]

print(tools)

[{'type': 'function', 'function': {'name': 'get_slang_definition', 'description': 'Get the definition of a slang word', 'parameters': {'type': 'object', 'properties': {'slang_word': {'type': 'string', 'description': 'The slang word to get the definition of'}}, 'required': ['slang_word'], 'additionalProperties': False}}}]


### Get OpenAI to use the tool

In [37]:
def chat(message, history):
    # add user message to history
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [
        {"role": "user", "content": message}
    ]

    # First model call – it may decide to call a tool
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=100,
        tools=tools,
    )

    assistant_message = response.choices[0].message

    # If the model decided to call a tool
    if assistant_message.tool_calls:
        tool_call = assistant_message.tool_calls[0]

        # Build an assistant message that contains the tool call
        assistant_tool_call_msg = {
            "role": "assistant",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments,
                    },
                }
            ],
        }

        # Run the tool locally and get a tool message
        tool_result_msg = handle_tool_call(tool_call)

        # Extend the conversation with the tool call + result
        messages.extend([assistant_tool_call_msg, tool_result_msg])

        # Call the model again so it can use the tool output
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

    return response.choices[0].message.content

    

In [38]:
def handle_tool_call(tool_call):
    """Execute a tool call from the model and return a tool message."""
    if tool_call.function.name == "get_slang_definition":
        arguments = json.loads(tool_call.function.arguments)
        slang_word = arguments.get("slang_word")
        slang_definition = get_slang_definition(slang_word)

        # Tool messages MUST include tool_call_id
        response = {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": slang_definition,
        }
        return response

    # Unknown tool – return no-op
    return {"role": "tool", "tool_call_id": tool_call.id, "content": "No-op"}


In [39]:
gr.ChatInterface(fn=chat, type="messages").launch()

Running on local URL:  http://127.0.0.1:7876

To create a public link, set `share=True` in `launch()`.


Getting definition for bussin
{'role': 'system', 'content': '\nYou are a helpful assistant for a GenZ chatbot.\nBe friendly and engaging.\nGive it an example of how to use the slang word, only when nessesary.\n'}
{'role': 'user', 'content': 'hi'}
{'role': 'assistant', 'content': 'Hey! What’s up? How can I help you today? 😊'}
{'role': 'user', 'content': 'what does bussin mean'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_rJ04bo4oZHsvSUnMuNEv0GJK', function=Function(arguments='{"slang_word":"bussin"}', name='get_slang_definition'), type='function')])
{'role': 'tool', 'content': 'bussin means Being very successful or popular.'}


Traceback (most recent call last):
  File "/Users/sanka/Projects/genz_chatbot/.venv/lib/python3.9/site-packages/gradio/queueing.py", line 536, in process_events
    response = await route_utils.call_process_api(
  File "/Users/sanka/Projects/genz_chatbot/.venv/lib/python3.9/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
  File "/Users/sanka/Projects/genz_chatbot/.venv/lib/python3.9/site-packages/gradio/blocks.py", line 1935, in process_api
    result = await self.call_function(
  File "/Users/sanka/Projects/genz_chatbot/.venv/lib/python3.9/site-packages/gradio/blocks.py", line 1518, in call_function
    prediction = await fn(*processed_input)
  File "/Users/sanka/Projects/genz_chatbot/.venv/lib/python3.9/site-packages/gradio/utils.py", line 793, in async_wrapper
    response = await f(*args, **kwargs)
  File "/Users/sanka/Projects/genz_chatbot/.venv/lib/python3.9/site-packages/gradio/chat_interface.py", line 623, in 